# Grid Search dedicado para deteccao de anomalias (AP combinado)

Notebook focado apenas em busca de hiperparametros com objetivo de maximizar Average Precision.

Escopo:
- preprocessamento dos datasets estabel, lav_temp e pecu
- avaliacao individual por dataset e avaliacao combinada (max score entre datasets)
- salvamento iterativo de resultados e checkpoints a cada combinacao
- compactacao e download dos resultados no final
- encerramento do runtime no Google Colab

In [1]:
IN_COLAB = False
try:
    from google.colab import files, runtime
    from google.colab import drive
    IN_COLAB = True
except Exception:
    pass

if IN_COLAB:
    # Montando o Google Drive para salvar os arquivos de resultado
    drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# Dependencias (Colab): instala somente se faltar algum pacote
import importlib

required_modules = [
    "pyod",
    "pandas",
    "numpy",
    "sklearn",
    "pyarrow",
]

missing = []
for mod in required_modules:
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(mod)

if missing:
    print(f"Instalando pacotes ausentes: {missing}")
    %pip install -q pyod combo pyarrow seaborn
else:
    print("Dependencias principais ja disponiveis.")

Dependencias principais ja disponiveis.


In [2]:
import os
import sys
import json
import math
import time
import shutil
import hashlib
import itertools
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score

from pyod.models.iforest import IForest
from pyod.models.hbos import HBOS
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD
from pyod.models.lof import LOF
from pyod.models.inne import INNE
from pyod.models.loda import LODA
from pyod.models.cblof import CBLOF

try:
    import torch
    AE_DEVICE = (
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
except Exception:
    AE_DEVICE = "cpu"

IN_COLAB = False
try:
    from google.colab import files, runtime
    IN_COLAB = True
except Exception:
    pass

# O upload do arquivo deve ser feito antes de importar o encoder, para garantir que o arquivo esteja
#  disponivel no caminho esperado. Se o arquivo ja estiver presente, o upload sera ignorado.
from scripts.encoder import AnomalyDetectionEncoder


KEY_COLS = ["V010100", "NUM_QUADRA", "NUM_FACE", "V010800"]
CONTAMINATION = 0.0687
SAMPLE_SIZE = 100_000
RANDOM_STATE = 42
DATASET_FILES = {
    "estabel": "df_estabel_final.parquet",
    "lav_temp": "df_lav_temp_final.parquet",
    "pecu": "df_pecu_final.parquet",
}
RESUME_IF_AVAILABLE = True
ENABLE_AUTOENCODER = True
CHECKPOINT_DIR = Path("/content/drive/MyDrive/Trabalho/TR COAGRO/censo agro 2026/notebooks/gridsearch_outputs/gs_ap_combined_100k_20260518_231005")  # Ex.: "gridsearch_outputs/gs_ap_combined_100k_20260517_142956"

LABELS_PATH = Path("experimentos") / "ground_truth.csv"
DATA_DIR = Path("experimentos") / "abordagem1"
OUTPUT_BASE = Path("/content/drive/MyDrive/Trabalho/TR COAGRO/censo agro 2026/notebooks/gridsearch_outputs") if IN_COLAB else Path("gridsearch_outputs")

checkpoint_dir_str = str(CHECKPOINT_DIR).strip() if CHECKPOINT_DIR is not None else ""
if checkpoint_dir_str:
    RUN_DIR = Path(checkpoint_dir_str).expanduser()
    RUN_LABEL = RUN_DIR.name
    RESTORE_FROM_CHECKPOINT = True
    if not RUN_DIR.exists():
        raise FileNotFoundError(f"CHECKPOINT_DIR informado nao existe: {RUN_DIR}")
else:
    RESTORE_FROM_CHECKPOINT = False
    RUN_LABEL = f"gs_ap_combined_{SAMPLE_SIZE//1000}k_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    RUN_DIR = OUTPUT_BASE / RUN_LABEL

for sub in ["scores", "combined_scores", "snapshots", "logs"]:
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f"RUN_DIR: {RUN_DIR}")
print(f"IN_COLAB: {IN_COLAB}")
print(f"AE_DEVICE: {AE_DEVICE}")
print(f"RESTORE_FROM_CHECKPOINT: {RESTORE_FROM_CHECKPOINT}")
if RESTORE_FROM_CHECKPOINT:
    print(f"CHECKPOINT_DIR: {RUN_DIR}")
else:
    print("CHECKPOINT_DIR vazio: iniciando execucao do zero.")

RUN_DIR: /content/drive/MyDrive/Trabalho/TR COAGRO/censo agro 2026/notebooks/gridsearch_outputs/gs_ap_combined_100k_20260518_231005
IN_COLAB: True
AE_DEVICE: cpu
RESTORE_FROM_CHECKPOINT: True
CHECKPOINT_DIR: /content/drive/MyDrive/Trabalho/TR COAGRO/censo agro 2026/notebooks/gridsearch_outputs/gs_ap_combined_100k_20260518_231005


In [3]:
def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)

def safe_ap(y_true, scores):
    y = np.asarray(y_true)
    s = np.asarray(scores)
    if len(y) == 0 or np.unique(y).size < 2:
        return float("nan")
    return float(average_precision_score(y, s))

def safe_auc(y_true, scores):
    y = np.asarray(y_true)
    s = np.asarray(scores)
    if len(y) == 0 or np.unique(y).size < 2:
        return float("nan")
    return float(roc_auc_score(y, s))

def preprocessar_dataset(nome: str, parquet_path: Path, labels_df: pd.DataFrame) -> dict:
    df = pd.read_parquet(parquet_path)

    for col in KEY_COLS:
        if col not in df.columns:
            raise KeyError(f"Coluna {col} nao encontrada no dataset {nome}.")
        df[col] = df[col].astype(str)

    df = df.merge(labels_df[KEY_COLS + ["is_anomaly"]], on=KEY_COLS, how="left")
    df["is_anomaly"] = df["is_anomaly"].fillna(0).astype(int)

    n_target = min(SAMPLE_SIZE, len(df))
    if n_target < len(df):
        sampled = []
        for label, grp in df.groupby("is_anomaly"):
            n_grp = max(1, round(n_target * len(grp) / len(df)))
            n_grp = min(n_grp, len(grp))
            sampled.append(grp.sample(n=n_grp, random_state=RANDOM_STATE))
        df = pd.concat(sampled, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    y = df["is_anomaly"].values
    if np.unique(y).size < 2:
        raise ValueError(f"Dataset {nome} ficou com apenas uma classe apos amostragem.")

    key_df = df[KEY_COLS].copy()
    X = df.drop(columns=KEY_COLS + ["is_anomaly"], errors="ignore")

    X_train, X_test, y_train, y_test, key_train, key_test = train_test_split(
        X,
        y,
        key_df,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y,
    )

    encoder = AnomalyDetectionEncoder(key_cols=[])
    encoder.fit(X_train)
    X_train_enc = encoder.transform(X_train)
    X_test_enc = encoder.transform(X_test)

    imputer = SimpleImputer(strategy="constant", fill_value=0)
    X_train_enc = imputer.fit_transform(X_train_enc)
    X_test_enc = imputer.transform(X_test_enc)

    return {
        "X_train": X_train_enc,
        "X_test": X_test_enc,
        "y_train": y_train,
        "y_test": pd.Series(y_test, name="is_anomaly"),
        "key_test": key_test.reset_index(drop=True),
    }

def make_param_combinations(param_grid: dict):
    keys = list(param_grid.keys())
    values = list(param_grid.values())
    for combo in itertools.product(*values):
        yield dict(zip(keys, combo))

def build_combined_df(per_ds_score_dfs: list[pd.DataFrame]) -> pd.DataFrame:
    if not per_ds_score_dfs:
        return pd.DataFrame()

    merged = per_ds_score_dfs[0].copy()
    for part in per_ds_score_dfs[1:]:
        merged = merged.merge(part, on=KEY_COLS, how="outer")

    score_cols = [c for c in merged.columns if c.startswith("score_")]
    y_cols = [c for c in merged.columns if c.startswith("y_true_")]

    merged["combined_score"] = merged[score_cols].max(axis=1, skipna=True)
    if y_cols:
        merged["y_true"] = merged[y_cols].max(axis=1, skipna=True).fillna(0).astype(int)
    else:
        merged["y_true"] = 0

    out_cols = KEY_COLS + ["combined_score", "y_true"]
    return merged[out_cols].dropna(subset=["combined_score"]).reset_index(drop=True)

In [4]:
if not LABELS_PATH.exists():
    raise FileNotFoundError(f"Arquivo de labels nao encontrado: {LABELS_PATH}")

labels_df = pd.read_csv(LABELS_PATH)
required_cols = KEY_COLS + ["is_anomaly"]
missing = [c for c in required_cols if c not in labels_df.columns]
if missing:
    raise ValueError(f"Colunas ausentes em labels: {missing}")

for c in KEY_COLS:
    labels_df[c] = labels_df[c].astype(str)
labels_df["is_anomaly"] = labels_df["is_anomaly"].astype(int)

datasets = {}
prep_rows = []

for ds_name, filename in DATASET_FILES.items():
    parquet_path = DATA_DIR / filename
    if not parquet_path.exists():
        print(f"[SKIP] Dataset ausente: {ds_name} -> {parquet_path}")
        prep_rows.append({"dataset": ds_name, "status": "missing", "path": str(parquet_path)})
        continue

    try:
        t0 = time.time()
        datasets[ds_name] = preprocessar_dataset(ds_name, parquet_path, labels_df)
        elapsed = time.time() - t0
        prep_rows.append({
            "dataset": ds_name,
            "status": "ok",
            "path": str(parquet_path),
            "n_train": int(datasets[ds_name]["X_train"].shape[0]),
            "n_test": int(datasets[ds_name]["X_test"].shape[0]),
            "pos_rate_test": float(datasets[ds_name]["y_test"].mean()),
            "time_s": round(elapsed, 3),
        })
        print(f"[OK] {ds_name}: train={datasets[ds_name]['X_train'].shape}, test={datasets[ds_name]['X_test'].shape}")
    except Exception as e:
        prep_rows.append({"dataset": ds_name, "status": "error", "path": str(parquet_path), "error": str(e)})
        print(f"[ERROR] {ds_name}: {e}")

prep_df = pd.DataFrame(prep_rows)
prep_df.to_csv(RUN_DIR / "snapshots" / "dataset_preprocessing_status.csv", index=False)

if not datasets:
    raise RuntimeError("Nenhum dataset valido foi preprocessado.")

save_json(
    {
        "run_dir": str(RUN_DIR),
        "run_label": RUN_LABEL,
        "restore_from_checkpoint": RESTORE_FROM_CHECKPOINT,
        "checkpoint_dir": str(RUN_DIR) if RESTORE_FROM_CHECKPOINT else "",
        "labels_path": str(LABELS_PATH),
        "data_dir": str(DATA_DIR),
        "dataset_files": DATASET_FILES,
        "available_datasets": list(datasets.keys()),
        "contamination": CONTAMINATION,
        "sample_size": SAMPLE_SIZE,
        "random_state": RANDOM_STATE,
    },
    RUN_DIR / "config.json",
)

prep_df

[OK] estabel: train=(80000, 321), test=(20000, 321)
[OK] lav_temp: train=(80000, 336), test=(20000, 336)
[OK] pecu: train=(73105, 545), test=(18277, 545)


,dataset,status,path,n_train,n_test,pos_rate_test,time_s
0,estabel,ok,experimentos/abordagem1/df_estabel_final.parquet,80000,20000,0.068700,4.929
1,lav_temp,ok,experimentos/abordagem1/df_lav_temp_final.parquet,80000,20000,0.069850,6.266
2,pecu,ok,experimentos/abordagem1/df_pecu_final.parquet,73105,18277,0.065602,7.161


In [5]:
# MODEL_SPECS = {
#     "IForest": {
#         "cls": IForest,
#         "fixed": {"random_state": RANDOM_STATE, "n_jobs": -1},
#         "grid": {
#             "n_estimators": [100, 200, 400],
#             "max_samples": ["auto", 256],
#             "max_features": [0.6, 0.8, 1.0],
#             "bootstrap": [False, True],
#         },
#     },
#     "HBOS": {
#         "cls": HBOS,
#         "fixed": {},
#         "grid": {
#             "n_bins": [20, 50, 80],
#             "alpha": [0.05, 0.1, 0.3],
#             "tol": [0.3, 0.5],
#         },
#     },
#     "ECOD": {
#         "cls": ECOD,
#         "fixed": {"n_jobs": -1},
#         "grid": {},
#     },
#     "COPOD": {
#         "cls": COPOD,
#         "fixed": {"n_jobs": -1},
#         "grid": {},
#     },
#     "LOF": {
#         "cls": LOF,
#         "fixed": {"n_jobs": -1},
#         "grid": {
#             "n_neighbors": [10, 20, 35],
#             "leaf_size": [20, 30],
#             "p": [1, 2],
#         },
#     },
#     "CBLOF": {
#         "cls": CBLOF,
#         "fixed": {"random_state": RANDOM_STATE},
#         "grid": {
#             "n_clusters": [6, 8, 12],
#             "alpha": [0.7, 0.9],
#             "beta": [3, 5],
#             "use_weights": [False, True],
#         },
#     },
#     "INNE": {
#         "cls": INNE,
#         "fixed": {"random_state": RANDOM_STATE},
#         "grid": {
#             "n_estimators": [100, 200, 400],
#             "max_samples": ["auto", 256],
#         },
#     },
#     "LODA": {
#         "cls": LODA,
#         "fixed": {},
#         "grid": {
#             "n_bins": [20, 50, 100],
#             "n_random_cuts": [50, 100],
#         },
#     },
# }

# if ENABLE_AUTOENCODER:
#     from pyod.models.auto_encoder import AutoEncoder
#     MODEL_SPECS["AutoEncoder"] = {
#         "cls": AutoEncoder,
#         "fixed": {"random_state": RANDOM_STATE, "device": AE_DEVICE},
#         "grid": {
#             "hidden_neuron_list": [[64, 32, 64], [64, 32, 32, 64]],
#             "epoch_num": [20, 40],
#             "batch_size": [32, 64],
#             "lr": [1e-3],
#         },
#     }

# plan_rows = []
# for model_name, spec in MODEL_SPECS.items():
#     n_combos = math.prod([len(v) for v in spec["grid"].values()])
#     plan_rows.append({"model": model_name, "combinations": int(n_combos)})

# plan_df = pd.DataFrame(plan_rows).sort_values("combinations", ascending=False).reset_index(drop=True)
# plan_df.to_csv(RUN_DIR / "snapshots" / "grid_plan.csv", index=False)

# print(f"Total de modelos no grid: {len(MODEL_SPECS)}")
# print(f"Total de combinacoes planejadas: {int(plan_df['combinations'].sum())}")
# plan_df

In [6]:
MODEL_SPECS = {
    "IForest": {
        "cls": IForest,
        "fixed": {"random_state": RANDOM_STATE, "n_jobs": -1},
        "grid": {
            "n_estimators": [100, 200, 400],
            "max_samples": [256, 512, 1024],
            "max_features": [0.1, 0.4, 0.6, 0.8, 1.0],
            "bootstrap": [False, True],
        },
    },
    "LODA": {
        "cls": LODA,
        "fixed": {},
        "grid": {
            "n_bins": [20, 50, 100, 200, 400],
            "n_random_cuts": [20, 50, 100, 200],
        },
    },
    "HBOS": {
        "cls": HBOS,
        "fixed": {},
        "grid": {
            "n_bins": [5, 10, 15, 20],
            "alpha": [0.05, 0.1, 0.3, 0.5, 0.8, 1.0],
            "tol": [0.3, 0.5, 0.6, 0.7],
        },
    },
    "CBLOF": {
        "cls": CBLOF,
        "fixed": {"random_state": RANDOM_STATE},
        "grid": {
            "n_clusters": [2, 4, 6, 8, 12],
            "alpha": [0.3, 0.7, 0.9, 1],
            "beta": [3, 5],
            "use_weights": [False],
        },
    },
    "INNE": {
        "cls": INNE,
        "fixed": {"random_state": RANDOM_STATE},
        "grid": {
            "n_estimators": [50, 100, 200, 400, 800],
            "max_samples": [256, 512, 1024],
        },
    },
    "LOF": {
        "cls": LOF,
        "fixed": {"n_jobs": -1},
        "grid": {
            "n_neighbors": [10, 20, 35, 50, 75],
            "leaf_size": [10, 20, 30, 50],
            "p": [1],
        },
    },
    
}

if ENABLE_AUTOENCODER:
    from pyod.models.auto_encoder import AutoEncoder
    MODEL_SPECS["AutoEncoder"] = {
        "cls": AutoEncoder,
        "fixed": {"random_state": RANDOM_STATE, "device": AE_DEVICE},
        "grid": {
            "hidden_neuron_list": [[64, 32, 64], [64, 32, 32, 64], [64, 32, 32, 32, 64]],
            "epoch_num": [20, 40, 60, 80],
            "batch_size": [32, 64, 128],
            "lr": [1e-4, 1e-3, 1e-2, 1e-1],
        },
    }

plan_rows = []
for model_name, spec in MODEL_SPECS.items():
    n_combos = math.prod([len(v) for v in spec["grid"].values()])
    plan_rows.append({"model": model_name, "combinations": int(n_combos)})

plan_df = pd.DataFrame(plan_rows).sort_values("combinations", ascending=False).reset_index(drop=True)
plan_df.to_csv(RUN_DIR / "snapshots" / "grid_plan.csv", index=False)

print(f"Total de modelos no grid: {len(MODEL_SPECS)}")
print(f"Total de combinacoes planejadas: {int(plan_df['combinations'].sum())}")
plan_df

Total de modelos no grid: 7
Total de combinacoes planejadas: 425


,model,combinations
0,AutoEncoder,144
1,HBOS,96
2,IForest,90
3,CBLOF,40
4,LODA,20
5,LOF,20
6,INNE,15


In [7]:
results_file = RUN_DIR / "results_iterative.csv"
checkpoint_file = RUN_DIR / "checkpoint_state.json"
ranking_snapshot_file = RUN_DIR / "snapshots" / "ranking_snapshot.csv"
best_by_model_file = RUN_DIR / "snapshots" / "best_by_model_iterative.csv"

all_jobs = []
for model_name, spec in MODEL_SPECS.items():
    for params in make_param_combinations(spec["grid"]):
        all_jobs.append((model_name, params))

checkpoint_state = {}
if RESUME_IF_AVAILABLE and checkpoint_file.exists():
    try:
        with open(checkpoint_file, "r", encoding="utf-8") as f:
            checkpoint_state = json.load(f)
    except Exception as e:
        print(f"[WARN] Falha ao ler checkpoint_state.json: {e}")

completed_keys = set()
if RESUME_IF_AVAILABLE and results_file.exists():
    try:
        prev = pd.read_csv(results_file)
    except pd.errors.EmptyDataError:
        prev = pd.DataFrame()

    if {"model", "combo_id", "status"}.issubset(prev.columns):
        done = prev[prev["status"] == "ok"]
        completed_keys = set(done["model"].astype(str) + "::" + done["combo_id"].astype(str))

print(f"Jobs planejados: {len(all_jobs)}")
print(f"Jobs ja concluidos (resume): {len(completed_keys)}")
if checkpoint_state:
    print(
        f"Checkpoint anterior detectado: idx={checkpoint_state.get('last_processed_job_index')} ",
        f"model={checkpoint_state.get('last_model')} combo={checkpoint_state.get('last_combo_id')} ",
        f"status={checkpoint_state.get('last_status')}"
    )
elif RESTORE_FROM_CHECKPOINT:
    print("CHECKPOINT_DIR informado sem checkpoint_state.json valido; retomada usando apenas results_iterative.csv.")

for idx, (model_name, params) in enumerate(all_jobs, start=1):
    param_json = json.dumps(params, sort_keys=True, default=str)
    combo_id = hashlib.md5(param_json.encode("utf-8")).hexdigest()[:12]
    job_key = f"{model_name}::{combo_id}"

    if job_key in completed_keys:
        continue

    spec = MODEL_SPECS[model_name]
    model_cls = spec["cls"]
    fixed = spec["fixed"]

    row = {
        "timestamp": datetime.now().isoformat(),
        "job_index": idx,
        "job_total": len(all_jobs),
        "model": model_name,
        "combo_id": combo_id,
        "params_json": param_json,
        "status": "ok",
    }

    print(f"[{idx:04d}/{len(all_jobs):04d}] {model_name} | combo={combo_id}")

    t0_job = time.time()
    per_ds_score_dfs = []
    ap_values = []

    try:
        for ds_name, ds in datasets.items():
            model_kwargs = {**fixed, **params}
            model = model_cls(contamination=CONTAMINATION, **model_kwargs)

            t0_fit = time.time()
            model.fit(ds["X_train"])
            scores = model.decision_function(ds["X_test"])
            fit_time = time.time() - t0_fit

            y_test = ds["y_test"].values
            ap_ds = safe_ap(y_test, scores)
            auc_ds = safe_auc(y_test, scores)

            row[f"ap_{ds_name}"] = ap_ds
            row[f"auc_{ds_name}"] = auc_ds
            row[f"fit_time_s_{ds_name}"] = round(float(fit_time), 4)
            ap_values.append(ap_ds)

            score_df = ds["key_test"].copy()
            score_df[f"score_{ds_name}"] = scores
            score_df[f"y_true_{ds_name}"] = y_test
            per_ds_score_dfs.append(score_df)

            score_out = RUN_DIR / "scores" / f"{model_name}__{combo_id}__{ds_name}.parquet"
            score_df.to_parquet(score_out, index=False)

        combined_df = build_combined_df(per_ds_score_dfs)
        combined_ap = safe_ap(combined_df["y_true"].values, combined_df["combined_score"].values) if not combined_df.empty else float("nan")
        combined_auc = safe_auc(combined_df["y_true"].values, combined_df["combined_score"].values) if not combined_df.empty else float("nan")

        row["ap_combined"] = combined_ap
        row["auc_combined"] = combined_auc

        if not combined_df.empty:
            combined_out = RUN_DIR / "combined_scores" / f"{model_name}__{combo_id}__combined.parquet"
            combined_df.to_parquet(combined_out, index=False)

        valid_aps = [x for x in ap_values if pd.notna(x)]
        row["ap_individual_mean"] = float(np.mean(valid_aps)) if valid_aps else float("nan")
        row["objective_ap"] = row["ap_combined"] if pd.notna(row.get("ap_combined")) else row["ap_individual_mean"]
        row["total_job_time_s"] = round(float(time.time() - t0_job), 4)

    except Exception as e:
        row["status"] = "error"
        row["error"] = str(e)
        row["total_job_time_s"] = round(float(time.time() - t0_job), 4)
        print(f"  -> erro: {e}")

    pd.DataFrame([row]).to_csv(results_file, mode="a", header=not results_file.exists(), index=False)

    current = pd.read_csv(results_file)
    if "objective_ap" in current.columns:
        ranking = current[current["status"] == "ok"].sort_values("objective_ap", ascending=False)
    else:
        ranking = current[current["status"] == "ok"]

    ranking.to_csv(ranking_snapshot_file, index=False)

    if not ranking.empty:
        best_by_model = ranking.groupby("model", as_index=False).first()
        best_by_model.to_csv(best_by_model_file, index=False)

    save_json(
        {
            "last_processed_job_index": idx,
            "last_model": model_name,
            "last_combo_id": combo_id,
            "last_status": row["status"],
            "updated_at": datetime.now().isoformat(),
            "results_file": str(results_file),
        },
        checkpoint_file,
    )

print("Grid search finalizado.")
print(f"Arquivo iterativo principal: {results_file}")

Jobs planejados: 425
Jobs ja concluidos (resume): 386
Checkpoint anterior detectado: idx=412  model=AutoEncoder combo=711ad1aaadfa  status=ok
[0131/0425] HBOS | combo=ae72e78af583
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0132/0425] HBOS | combo=452418a50b11
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0133/0425] HBOS | combo=6d860d5e20fa
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0134/0425] HBOS | combo=665cc6c81cc5
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0155/0425] HBOS | combo=209313b8a59f
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0156/0425] HBOS | combo=17278f63f60f
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0157/0425] HBOS | combo=3f0eed680c6b
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0158/0425] HBOS | combo=7fe349befb0b
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0179/0425] HBOS | combo=cf9e9f887dd1
  -> erro: alpha is set to 1.0. Not 

[0205/0425] HBOS | combo=b4837306c24e
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0206/0425] HBOS | combo=70f852bf176d
  -> erro: alpha is set to 1.0. Not in the range of (0, 1).
[0213/0425] CBLOF | combo=0a0a0ed426c5
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0214/0425] CBLOF | combo=7c648b122a37
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0221/0425] CBLOF | combo=a7a5d1ee7df9
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0222/0425] CBLOF | combo=1f9524cb2e54
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0229/0425] CBLOF | combo=fd31851b1496
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0230/0425] CBLOF | combo=857790c14f6d
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0237/0425] CBLOF | combo=112acb32d6c7
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0238/0425] CBLOF | combo=ba1e4a6a6a03
  -> erro: alpha is set to 1. Not in the range of (0, 1).
[0245/0425] CBLOF | combo=17

Training: 100%|██████████| 60/60 [06:37<00:00,  6.63s/it]


[0414/0425] AutoEncoder | combo=2c8893aa2674


Training: 100%|██████████| 80/80 [24:05<00:00, 18.06s/it]


...

[0424/0425] AutoEncoder | combo=0e52e1bfb33a


Training: 100%|██████████| 80/80 [08:25<00:00,  6.32s/it]


[0425/0425] AutoEncoder | combo=33c20607aa4b


Training: 100%|██████████| 80/80 [06:03<00:00,  4.54s/it]


Grid search finalizado.
Arquivo iterativo principal: /content/drive/MyDrive/Trabalho/TR COAGRO/censo agro 2026/notebooks/gridsearch_outputs/gs_ap_combined_100k_20260518_231005/results_iterative.csv


In [8]:
if not results_file.exists():
    raise FileNotFoundError("results_iterative.csv nao foi criado.")

results_df = pd.read_csv(results_file)
ok_df = results_df[results_df["status"] == "ok"].copy()

if ok_df.empty:
    print("Nenhuma combinacao valida foi concluida com status ok.")
else:
    ok_df = ok_df.sort_values("objective_ap", ascending=False).reset_index(drop=True)
    ok_df.to_csv(RUN_DIR / "final_ranking.csv", index=False)

    best_global = ok_df.iloc[0].to_dict()
    best_by_model = ok_df.groupby("model", as_index=False).first()
    best_by_model.to_csv(RUN_DIR / "best_by_model.csv", index=False)

    summary = {
        "run_dir": str(RUN_DIR),
        "n_total_rows": int(results_df.shape[0]),
        "n_ok_rows": int(ok_df.shape[0]),
        "n_error_rows": int((results_df["status"] == "error").sum()),
        "best_global": best_global,
    }
    save_json(summary, RUN_DIR / "summary.json")

    print("Top 10 configuracoes por AP objetivo (AP combinado como prioridade):")
    display_cols = [
        "model", "combo_id", "objective_ap", "ap_combined", "ap_individual_mean"
    ] + [c for c in ok_df.columns if c.startswith("ap_") and c not in ["ap_combined", "ap_individual_mean"]]
    print(ok_df[display_cols].head(10).to_string(index=False))

Top 10 configuracoes por AP objetivo (AP combinado como prioridade):
      model     combo_id  objective_ap  ap_combined  ap_individual_mean          ap_estabel  ap_lav_temp  ap_pecu
AutoEncoder eb95f118b8a5      0.134932     0.134932            0.138805 0.12112435975292855     0.139559 0.155731
AutoEncoder 7dd2b90217ee      0.134928     0.134928            0.138524 0.12102444586075142     0.138261 0.156287
AutoEncoder 192a9b77785a      0.134860     0.134860            0.138464 0.12037769004828489     0.139302 0.155713
AutoEncoder 914aa9524b53      0.134747     0.134747            0.139085 0.12029113537746593     0.140678 0.156287
AutoEncoder 72abad00bbdd      0.134672     0.134672            0.138808 0.12126457009647458     0.139774 0.155385
AutoEncoder 8fcf29f81374      0.134617     0.134617            0.138596 0.12126749311225908     0.139524 0.154995
AutoEncoder e5dba811e8bb      0.134533     0.134533            0.138367  0.1195378964972813     0.139860 0.155704
AutoEncoder 33e3639

In [9]:
if IN_COLAB:
    print("Finalizando runtime para preservar recurso...")
    runtime.unassign()

Finalizando runtime para preservar recurso...
